# Week 17, Day 2: Maximum likelihood: fitting rates, tuning curves and GLMs

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The mean count maximises the likelihood

Write `poisson_ll(lam, k)` returning the sum of k log(lam) − lam (dropping the constant). Evaluate it on a fine grid of rates for 25 simulated counts and confirm that the maximum is at the mean count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2008)
counts = rng.poisson(6.0, 25)

def poisson_ll(lam, k):
    return ...

grid = np.arange(0.5, 15, 0.01)
ll = np.array([poisson_ll(l, counts) for l in grid])
lam_grid_best = grid[np.argmax(ll)]
print(f"grid maximum {lam_grid_best:.2f}, mean count {counts.mean():.2f}")
plt.plot(grid, ll); plt.axvline(counts.mean(), color="k", ls="--"); plt.xlabel("rate"); plt.ylabel("log-likelihood"); plt.show()

In [ ]:
import numpy as np
def test_ll():
    "poisson_ll is the Poisson log-likelihood, up to a constant"
    k = np.array([2, 5, 3])
    assert abs((poisson_ll(4.0, k) - poisson_ll(3.0, k)) - (10 * np.log(4 / 3) - 3)) < 1e-9

def test_max():
    "the grid maximum agrees with the mean count to within the grid step"
    assert abs(lam_grid_best - counts.mean()) <= 0.01

test_ll()
test_max()
print("All checks passed")

### Exercise: Fit a tuning curve

Simulate 400 trials at random orientations from a neuron with log-rate 1 + 1.2 cos 2(θ − 60°). Build the design matrix X with columns (1, cos 2θ, sin 2θ), then run Newton's method from w = 0 until the step is smaller than 1e-10, counting the steps. Store `w_hat`, `n_steps`, `pref_hat` (degrees) and `strength_hat`.

In [ ]:
theta = rng.uniform(0, np.pi, 400)
log_rate = 1 + 1.2 * np.cos(2 * (theta - np.deg2rad(60)))
k = rng.poisson(np.exp(log_rate))
X = np.column_stack([np.ones_like(theta), np.cos(2 * theta), np.sin(2 * theta)])

w = np.zeros(3)
for n_steps in range(1, 50):
    lam = np.exp(X @ w)
    grad = ...
    hess = ...
    step = np.linalg.solve(hess, grad)
    w = w - step
    if np.abs(step).max() < 1e-10:
        break
w_hat = w
pref_hat = ...                                  # 0.5 * atan2(c, b), in degrees between 0 and 180
strength_hat = ...
print(f"Newton steps {n_steps};  weights {w_hat.round(3)};  preferred {pref_hat:.1f} deg, strength {strength_hat:.3f}")
tt = np.linspace(0, np.pi, 200); Xt = np.column_stack([np.ones_like(tt), np.cos(2 * tt), np.sin(2 * tt)])
plt.plot(np.rad2deg(theta), k, ".", alpha=0.4); plt.plot(np.rad2deg(tt), np.exp(Xt @ w_hat), "k"); plt.xlabel("orientation (deg)"); plt.ylabel("count"); plt.show()

In [ ]:
import numpy as np
def test_converged():
    "Newton's method converges: the gradient at the end is tiny"
    lam = np.exp(X @ w_hat)
    assert np.abs(X.T @ (k - lam)).max() < 1e-6

def test_steps():
    "it needs at most 12 Newton steps"
    assert n_steps <= 12

def test_recover():
    "the preferred orientation is within 5 degrees of 60, and the tuning strength within 0.15 of 1.2"
    assert abs(pref_hat - 60) < 5 and abs(strength_hat - 1.2) < 0.15

test_converged()
test_steps()
test_recover()
print("All checks passed")

### Exercise: Likelihood against least squares

A common shortcut is to fit log(count + 1) by ordinary least squares. For 300 simulated datasets of 60 trials each, estimate the preferred orientation both ways and store the squared errors (in degrees²) in `err_ml` and `err_ls`.

In [ ]:
def fit_ml(X, k, steps=30):
    w = np.zeros(X.shape[1])
    for _ in range(steps):
        lam = np.exp(X @ w)
        w = w - np.linalg.solve(-X.T @ (lam[:, None] * X), X.T @ (k - lam))
    return w

def pref(w): return np.mod(np.rad2deg(0.5 * np.arctan2(w[2], w[1])), 180)
def circ_err(a, b): d = np.mod(a - b, 180); return min(d, 180 - d) ** 2

err_ml, err_ls = [], []
for _ in range(300):
    th = rng.uniform(0, np.pi, 60)
    Xd = np.column_stack([np.ones_like(th), np.cos(2 * th), np.sin(2 * th)])
    kd = rng.poisson(np.exp(0.5 + 1.0 * np.cos(2 * (th - np.deg2rad(60)))))
    w_ls = ...                                   # least squares of log(kd + 1) on Xd
    err_ml.append(circ_err(pref(fit_ml(Xd, kd)), 60))
    err_ls.append(...)
print(f"mean squared error of the preferred orientation: maximum likelihood {np.mean(err_ml):.1f}, least squares {np.mean(err_ls):.1f} (deg^2)")

In [ ]:
import numpy as np
def test_both():
    "both methods are run on 300 simulated datasets"
    assert len(err_ml) == 300 and len(err_ls) == 300

def test_ml_better():
    "maximum likelihood recovers the preferred orientation with a smaller mean squared error than least squares on log(count + 1)"
    assert np.mean(err_ml) < np.mean(err_ls)

test_both()
test_ml_better()
print("All checks passed")